# Stage 1 — YOLOv11 Damage Detection — Colab T4 Training

Full 100-epoch run on a free Colab **T4 GPU**, logging to **Weights & Biases**.

## Before you start
1. **Runtime → Change runtime type → T4 GPU**.
2. Add two **Colab Secrets** (🔑 icon in the left sidebar), with *Notebook access* on:
   - `ROBOFLOW_API_KEY`
   - `WANDB_API_KEY`
3. Set `GIT_REPO_URL` below (if cloning) **or** upload the `stage1_detection/` folder (see the *Get the code* cell).

The dataset is re-downloaded from Roboflow inside Colab, so nothing large needs uploading.

In [ ]:
# 1. Confirm we have a GPU (should show a Tesla T4)
!nvidia-smi

In [ ]:
# 2. Install deps (subset only — keep Colab's pre-built CUDA torch, do NOT reinstall torch)
!pip install -q ultralytics roboflow wandb albumentations onnx onnxruntime onnxslim python-dotenv pyyaml
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'no gpu')

In [ ]:
# 3. Load API keys from Colab Secrets into the environment
import os
try:
    from google.colab import userdata
    os.environ['ROBOFLOW_API_KEY'] = userdata.get('ROBOFLOW_API_KEY')
    os.environ['WANDB_API_KEY']   = userdata.get('WANDB_API_KEY')
    print('Keys loaded from Colab Secrets.')
except Exception as e:
    from getpass import getpass
    print('Colab Secrets unavailable (', e, ') — falling back to manual entry.')
    os.environ['ROBOFLOW_API_KEY'] = getpass('ROBOFLOW_API_KEY: ')
    os.environ['WANDB_API_KEY']   = getpass('WANDB_API_KEY: ')
assert os.environ.get('ROBOFLOW_API_KEY') and os.environ.get('WANDB_API_KEY'), 'Missing a key'

In [ ]:
# 4. Mount Google Drive (so weights/checkpoints survive a disconnect)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/veh_agent_stage1'
os.makedirs(DRIVE_OUT, exist_ok=True)
print('Artifacts will be copied to:', DRIVE_OUT)

In [ ]:
# 5. Get the code (clone from your public GitHub repo).
GIT_REPO_URL = 'https://github.com/jgcarti123-cyber/Vehicle_Damage_Agent.git'

%cd /content
!rm -rf veh_agent && git clone {GIT_REPO_URL} veh_agent
%cd /content/veh_agent
!ls stage1_detection

In [ ]:
# 6. Download + remap(17->4) + stratified 80/10/10 split + validate
!python stage1_detection/dataset.py download
!python stage1_detection/dataset.py prepare
!python stage1_detection/dataset.py validate

In [ ]:
# 7. Full 100-epoch training on the T4 (device 0). W&B auto-enables from WANDB_API_KEY.
#    epochs/batch/etc. come from config.yaml; --device 0 overrides the config's 'mps'.
!python stage1_detection/train.py --device 0
#  If you hit VRAM limits, add: --batch 8
#  RESUME after a disconnect (weights persisted to Drive in cell 9):
#    !yolo detect train resume model=runs/detect/vehicle-damage-detection/yolo11s-baseline/weights/last.pt

In [ ]:
# 8. Evaluate on the held-out TEST split.
#    Run name/dir are read from config.yaml so this always matches the latest run.
import yaml
cfg = yaml.safe_load(open('stage1_detection/config.yaml'))
RUN = cfg['name']                                  # e.g. 'yolo11m-baseline'
RUN_DIR = f"runs/detect/{cfg['project']}/{RUN}"
BEST = f"{RUN_DIR}/weights/best.pt"
print('Evaluating run:', RUN, '->', BEST)
!python stage1_detection/evaluate.py --weights {BEST} --device 0

In [ ]:
# 9. Export ONNX + CPU latency benchmark, then copy artifacts to a per-run Drive subfolder
#    (each run saves to DRIVE_OUT/<run name>/ so runs never overwrite each other).
!python stage1_detection/export.py --weights {BEST} --benchmark
import shutil, os
DEST = os.path.join(DRIVE_OUT, RUN)
os.makedirs(DEST, exist_ok=True)
for f in ['weights/best.pt', 'weights/best.onnx', 'results.csv', 'results.png',
          'confusion_matrix.png', 'args.yaml']:
    src = os.path.join(RUN_DIR, f)
    if os.path.exists(src):
        shutil.copy2(src, DEST)
        print('saved', f)
print('All artifacts in:', DEST)